In [9]:
'''
Versão final 2.0
'''

import os
import re
import subprocess
import warnings
import numpy as np
import pandas as pd
from collections import Counter
from Bio import Entrez
from Bio import SeqIO
from Bio.Seq import Seq
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import subprocess

warnings.filterwarnings("ignore")

EMAIL = "marcela.leite@gmail.com.br"
OUTPUT_DIR = "pipelineI"
os.makedirs(OUTPUT_DIR, exist_ok=True)
Entrez.email = EMAIL

# ==========================================================
# QUERIES
# ==========================================================

POSITIVE_QUERY = r'''
("Solanum lycopersicum"[Organism] OR "Nicotiana benthamiana"[Organism] OR "Capsicum annuum"[Organism])
AND( TMV OR ToMV OR ToBRFV OR PMMoV OR TMGMV OR PVY OR PepYMV OR TEV OR PVA OR PVX OR TSWV OR GRSV  OR TCSV  OR CMV OR ToCV OR TICV OR AMV
OR "virus resistance" OR "antiviral" OR "RNA silencing" OR "RDR" OR "Dicer-like" OR "Argonaute" OR "AGO" OR "eIF4E" OR "R gene" OR "Sw-5" OR "Tm-2" OR "RCY1"
OR "Rx" OR "R protein"  OR "NLR" OR "NBS-LRR"  OR "NB-LRR" OR "TIR-NBS-LRR" OR "CC-NBS-LRR" OR "RLK" OR "RLP" OR "LRR receptor kinase"
OR "leucine rich repeat receptor kinase" OR "plant disease resistance protein" OR "disease resistance protein" OR "RPM1" OR "RPS2" OR "RPS4" OR "RPS5"
OR "RPP" OR "SNC1"  OR "ADR1" OR "NDR1" OR "EDS1" OR "PAD4" )
NOT ( partial OR fragment OR hypothetical OR predicted OR DAP-seq OR "transcription factor" OR DNA-binding OR microtom )
'''

NEGATIVE_QUERY = r'''
( "Solanum lycopersicum"[Organism] OR "Nicotiana benthamiana"[Organism] OR "Capsicum annuum"[Organism] )
AND ( "actin" OR "tubulin" OR "ribosomal protein" OR "ATP synthase" OR "photosystem" OR "elongation factor" OR "glycolysis" OR "metabolic enzyme" )
NOT ( "virus resistance"  OR "antiviral" OR "RNA silencing" OR "RDR" OR "Dicer-like" OR "Argonaute" OR "AGO" OR "eIF4E" OR "R gene" OR "Sw-5" OR "Tm-2"
OR "RCY1" OR "Rx" OR "R protein" OR "resistance" OR "NLR" OR "LRR" OR "immune" OR "virus" )
'''

ARABIDOPSIS_QUERY = '''
  "Arabidopsis thaliana"[Organism] AND
      (transcriptome OR RNA-Seq OR TSA OR leaf OR fruit OR root OR stress OR virus OR infected)
      NOT (genome OR chromosome OR scaffold OR contig OR partial OR fragment)
      AND 1500:30000[Sequence Length]
      '''

TUBEROSUM_QUERY = '''
    Solanum tuberosum [Organism]
AND (transcriptome OR RNA-Seq OR TSA OR leaf OR fruit OR root OR stress OR virus OR infected)
       NOT (genome OR chromosome OR scaffold OR contig OR partial OR fragment)
       AND 1500:30000[Sequence Length]
    '''

PHYSALIS_QUERY = '''
  "Physalis peruviana"[Organism] AND
      (transcriptome OR RNA-Seq OR TSA OR leaf OR fruit OR root OR stress OR virus OR infected)
      NOT (genome OR chromosome OR scaffold OR contig OR partial OR fragment)
      AND 1500:30000[Sequence Length]
      '''


# ==========================================================
# DOWNLOAD NCBI
# ==========================================================

def baixar_proteinas_ncbi(query, output_fasta, max_records=5000):
    print("\n================================")
    print("DOWNLOAD NCBI")
    print("================================")
    handle = Entrez.esearch(
        db="protein",
        term=query,
        retmax=max_records
    )
    record = Entrez.read(handle)
    ids = record["IdList"]
    print(f"Proteínas encontradas: {len(ids)}")
    if len(ids) == 0:
        return
    handle = Entrez.efetch(
        db="protein",
        id=ids,
        rettype="fasta",
        retmode="text"
    )
    with open(output_fasta, "w") as f:
        f.write(handle.read())
    print(f"Arquivo salvo: {output_fasta}")

# ==========================================================
# CD-HIT
# ==========================================================

def executar_cdhit(
    input_fasta,
    output_fasta,
    identity=0.5  #0.7
):
    cmd = [
        "cd-hit",
        "-i", input_fasta,
        "-o", output_fasta,
        "-c", str(identity),
        "-n", "5",
        "-M", "16000",
        "-T", "4"
    ]

    print("\nExecutando CD-HIT...")
    subprocess.run(cmd, check=True)
    print("CD-HIT concluído.")

# ==========================================================
# LER CLUSTERS CD-HIT
# ==========================================================
def limpar(seq_id):
    seq_id = seq_id.strip()
    seq_id = seq_id.split()[0]
    return seq_id

def ler_clusters_cdhit(cluster_file):
    clusters = {}
    cluster_id = None
    with open(cluster_file) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">Cluster"):
                cluster_id = line.replace(">Cluster ", "")
                clusters[cluster_id] = []
            else:
                seq_id = line.split(">")[1].split("...")[0]
                seq_id = limpar(seq_id)
                clusters[cluster_id].append(seq_id)
    return clusters

# ==========================================================
# DOWNLOAD TRANSCRIPTOMA
# ==========================================================

def baixar_transcriptoma(output_fasta, query):
    print("\n================================")
    print("DOWNLOAD TRANSCRIPTOMA")
    print("================================")
    handle = Entrez.esearch(
        db="nucleotide",
        term=query,
        retmax=10000
    )
    record = Entrez.read(handle)
    ids = record["IdList"]
    print(f"Transcritos encontrados: {len(ids)}")
    handle = Entrez.efetch(
        db="nucleotide",
        id=ids,
        rettype="fasta",
        retmode="text"
    )
    with open(output_fasta, "w") as f:
        f.write(handle.read())
    print(f"Arquivo salvo: {output_fasta}")

# ==========================================================
# ORF
# ==========================================================

def encontrar_maior_orf(seq):
    seq = seq.upper()
    maior = ""
    for frame in range(3):
        translated = str(Seq(seq[frame:]).translate())
        proteins = translated.split("*")
        for p in proteins:
            if len(p) > len(maior):
                maior = p
    return maior

# ==========================================================
# AAC
# ==========================================================

AMINO_ACIDOS = list("ACDEFGHIKLMNPQRSTVWY")

def calcular_aac(seq):
    seq = seq.upper()
    total = len(seq)
    if total == 0:
        return [0] * len(AMINO_ACIDOS)
    counts = Counter(seq)
    return [
        counts.get(aa, 0) / total
        for aa in AMINO_ACIDOS
    ]

# ==========================================================
# DIPEPTÍDEOS
# ==========================================================

DIPEPTIDES = [
    a + b
    for a in AMINO_ACIDOS
    for b in AMINO_ACIDOS
]

def calcular_dipeptideos(seq):
    seq = seq.upper()
    total = len(seq) - 1
    if total <= 0:
        return [0] * len(DIPEPTIDES)
    counts = Counter([
        seq[i:i+2]
        for i in range(total)
    ])
    return [
        counts.get(dp, 0) / total
        for dp in DIPEPTIDES
    ]

# ==========================================================
# FEATURES
# ==========================================================

def extrair_features_proteina(seq):
    seq = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", seq)
    if len(seq) < 50:
        return None
    features = []
    features.append(len(seq))
    features.extend(calcular_aac(seq))
    features.extend(calcular_dipeptideos(seq))
    return features

# ==========================================================
# DATASET
# ==========================================================

def criar_dataset(positive_fasta, negative_fasta):
    data = []
    columns = (
        ["protein_length"]
        +
        [f"AAC_{aa}" for aa in AMINO_ACIDOS]
        +
        [f"DIPEP_{dp}" for dp in DIPEPTIDES]
    )

    # POSITIVOS
    for record in SeqIO.parse(positive_fasta, "fasta"):
        seq = str(record.seq)
        feats = extrair_features_proteina(seq)
        if feats is None:
            continue

        row = feats + [1, record.id]
        data.append(row)

    # NEGATIVOS
    for record in SeqIO.parse(negative_fasta, "fasta"):
        seq = str(record.seq)
        feats = extrair_features_proteina(seq)
        if feats is None:
            continue
        row = feats + [0, record.id]
        data.append(row)
    df = pd.DataFrame(
        data,
        columns=columns + ["target", "id"]
    )

    return df, columns

# ==========================================================
# SPLIT POR HOMOLOGIA
# ==========================================================

def split_por_homologia(df):
    groups = df["cluster"]
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=42
    )
    train_idx, test_idx = next(
        splitter.split(df, groups=groups)
    )
    train_df = df.iloc[train_idx]
    test_df = df.iloc[test_idx]
    return train_df, test_df

# ==========================================================
# TREINAMENTO
# ==========================================================

def avaliar_modelo(nome, model, X_train, y_train, X_test, y_test):
    print("\n============================================================")
    print(nome)
    print("============================================================")
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:,1]
    preds = (probs >= 0.5).astype(int)
    print(classification_report(y_test, preds))
    roc = roc_auc_score(y_test, probs)
    pr = average_precision_score(y_test, probs)
    mcc = matthews_corrcoef(y_test, preds)
    f1 = f1_score(y_test, preds)
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)
    bal_acc = balanced_accuracy_score(y_test, preds)

    print("ROC-AUC          :", roc)
    print("PR-AUC           :", pr)
    print("MCC              :", mcc)
    print("F1               :", f1)
    print("Precision        :", precision)
    print("Recall           :", recall)
    print("Balanced Accuracy:", bal_acc)

    # ======================================================
    # MATRIZ DE CONFUSÃO
    # ======================================================
    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize=(5,5))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Negativo", "Positivo"]
    )
    disp.plot(cmap="Blues")
    plt.title(f"Matriz de Confusão - {nome}")
    plt.savefig(
        f"{OUTPUT_DIR}/{nome}_matriz_confusao.png",
        bbox_inches="tight"
    )
    plt.close()
    # ======================================================
    # CURVA ROC
    # ======================================================
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(6,6))
    plt.plot(
        fpr,
        tpr,
        label=f"AUC = {roc_auc:.4f}"
    )
    plt.plot([0,1], [0,1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"Curva ROC - {nome}")
    plt.legend(loc="lower right")
    plt.savefig(
        f"{OUTPUT_DIR}/{nome}_roc.png",
        bbox_inches="tight"
    )
    plt.close()
    return {
        "Modelo": nome,
        "ROC_AUC": roc,
        "F1": f1,
        "Precision": precision,
        "Recall": recall,
        "Balanced_Accuracy": bal_acc
    }, model
# ==========================================================
# VALIDAÇÃO MOTIFS
# ==========================================================

MOTIFS = {
    "P_LOOP": r"[A-Z]G[A-Z]{4}GK[ST]",
    "GLPL": r"GLPL",
    "MHD": r"MHD"
}

def validar_motifs(df):
    print("\nVALIDAÇÃO BIOLÓGICA\n")
    counts = {}
    for motif_name, pattern in MOTIFS.items():
        counts[motif_name] = 0
        for seq in df["protein_seq"]:
            if re.search(pattern, seq):
                counts[motif_name] += 1
    for k, v in counts.items():
        print(k, ":", v)


def rodar_transdecoder(fasta_nt):
    cmd1 = [
        "TransDecoder.LongOrfs",
        "-t",
        fasta_nt,
        "--output_dir",
        f"{OUTPUT_DIR}"
    ]
    with open(fasta_nt+"_output1.txt", "a", encoding="utf-8") as log_file:
        subprocess.run(cmd1, 
                       stdout=log_file, 
                       stderr=subprocess.STDOUT,
                       text=True)

    cmd2 = [
        "TransDecoder.Predict",
        "-t",
        fasta_nt,
        "--output_dir",
        f"{OUTPUT_DIR}"
    ]
    with open(fasta_nt+"_output2.txt", "a", encoding="utf-8") as log_file:
        subprocess.run(cmd2, 
                       stdout=log_file, 
                       stderr=subprocess.STDOUT,
                       text=True)
    pep_file = fasta_nt + ".transdecoder.pep"
    return pep_file


def predizer_transcritos(fasta_nt,model,nmodel):
    candidatos = []
    pep_file = rodar_transdecoder(fasta_nt)
    for record in SeqIO.parse(fasta_nt+".transdecoder.pep", "fasta"):
        protein = str(record.seq)
        feats = extrair_features_proteina(protein)
        if feats is None:
            continue
        X = scaler.transform([feats])
        prob = model.predict_proba(X)[0][1]
        candidatos.append({
            "id": record.id,
            "descricao": record.description,
            "protein_length": len(protein),
            "probabilidade_resistencia": prob,
            "protein_seq": protein
        })

    candidatos_df = pd.DataFrame(candidatos)
    candidatos_df = candidatos_df.sort_values(
        "probabilidade_resistencia",
        ascending=False
    )

    top = candidatos_df.head(20)
    print("\nTOP CANDIDATOS: "+fasta_nt+", MODELO: "+nmodel)
    print(top[[
        "id",
        "descricao",
        "protein_length",
        "probabilidade_resistencia"
    ]])

    validar_motifs(top)

    candidatos_df.to_csv(
        f"{OUTPUT_DIR}/predicoes.csv",
        index=False
    )
# ==========================================================
# MAIN
# ==========================================================

if __name__ == "__main__":
    POSITIVE_FASTA = f"{OUTPUT_DIR}/positive.fasta"
    NEGATIVE_FASTA = f"{OUTPUT_DIR}/negative.fasta"
    baixar_proteinas_ncbi(POSITIVE_QUERY,POSITIVE_FASTA)
    baixar_proteinas_ncbi(NEGATIVE_QUERY,NEGATIVE_FASTA)
    
    # ======================================================
    # CD-HIT
    # ======================================================

    executar_cdhit(
        POSITIVE_FASTA,
        f"{OUTPUT_DIR}/positive_nr.fasta",
        identity=0.7
    )

    executar_cdhit(
        NEGATIVE_FASTA,
        f"{OUTPUT_DIR}/negative_nr.fasta",
        identity=0.7
    )

    POSITIVE_FASTA = f"{OUTPUT_DIR}/positive_nr.fasta"
    NEGATIVE_FASTA = f"{OUTPUT_DIR}/negative_nr.fasta"

    # ======================================================
    # DATASET
    # ======================================================

    df, columns = criar_dataset(
        POSITIVE_FASTA,
        NEGATIVE_FASTA
    )

    df["id"] = df["id"].apply(limpar)
    # ======================================================
    # CLUSTERS
    # ======================================================

    clusters_pos = ler_clusters_cdhit(
        f"{OUTPUT_DIR}/positive_nr.fasta.clstr"
    )

    clusters_neg = ler_clusters_cdhit(
        f"{OUTPUT_DIR}/negative_nr.fasta.clstr"
    )

    cluster_map = {}

    for c, seqs in clusters_pos.items():
        for s in seqs:
            cluster_map[s] = f"POS_{c}"

    for c, seqs in clusters_neg.items():
        for s in seqs:
            cluster_map[s] = f"NEG_{c}"

    df["cluster"] = df["id"].map(cluster_map)

    print("\n Clusters ausentes: ")
    print(df["cluster"].isna().sum())
    df = df.dropna(subset=["cluster"])

    print("\nDataset:", df.shape)

    # ======================================================
    # SPLIT POR HOMOLOGIA
    # ======================================================

    train_df, test_df = split_por_homologia(df)
    X_train = train_df[columns]
    y_train = train_df["target"]
    X_test = test_df[columns]
    y_test = test_df["target"]
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # ======================================================
    # MODELOS
    # ======================================================

    rf = RandomForestClassifier(
        n_estimators=1000,
        max_depth = 12,
        min_samples_split=4,
        max_features='sqrt',
        random_state=42,
        class_weight="balanced",
        n_jobs = -1
    )

    xgb = XGBClassifier(
        n_estimators=1200,
        max_depth=5,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        min_child_weight=3,
        reg_alpha=0.5,
        reg_lambda=2,
        scale_pos_weight=1.2
    )

    svm = SVC(
        probability=True,
        kernel="rbf",
        class_weight="balanced",
        random_state=42,
        C=2,
        gamma='scale'
    )

    results = []

    r1, rf_model = avaliar_modelo(
        "Random Forest",
        rf,
        X_train,
        y_train,
        X_test,
        y_test
    )

    results.append(r1)

    r2, xgb_model = avaliar_modelo(
        "XGBoost",
        xgb,
        X_train,
        y_train,
        X_test,
        y_test
    )

    results.append(r2)

    r3, svm_model = avaliar_modelo(
        "SVM",
        svm,
        X_train,
        y_train,
        X_test,
        y_test
    )

    results.append(r3)
    print("\nCOMPARAÇÃO MODELOS")
    print(pd.DataFrame(results))

    results_df = pd.DataFrame(results)

    metrics = [
        "ROC_AUC",
        "F1",
        "Precision",
        "Recall",
        "Balanced_Accuracy"
    ]

    for metric in metrics:
        plt.figure(figsize=(8,5))
        plt.bar(
            results_df["Modelo"],
            results_df[metric]
        )

        plt.ylim(0, 1)
        plt.ylabel(metric)
        plt.title(f"Comparação dos Modelos - {metric}")
        for i, v in enumerate(results_df[metric]):
            plt.text(i, v + 0.01, f"{v:.3f}", ha='center')

        plt.savefig(
            f"{OUTPUT_DIR}/comparacao_{metric}.png",
            bbox_inches="tight"
        )
        plt.close()

    modelos = {
        "Random Forest":rf_model,
        "XGBoost":xgb_model,
        "SVM":svm_model
        }
    plt.figure(figsize=(7,7))
    for nome, model in modelos.items():
        probs = model.predict_proba(X_test)[:,1]
        fpr, tpr, _ = roc_curve(y_test, probs)
        roc_auc = auc(fpr, tpr)
        plt.plot(
            fpr,
            tpr,
            label=f"{nome} AUC={roc_auc:.3f}"
        )
    plt.plot([0,1],[0,1],'--')
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.title("Comparação ROC")
    plt.legend()
    plt.savefig(f"{OUTPUT_DIR}/roc_comparativa.png")
    plt.close()
    
    # ======================================================
    # PREVISÕES - BAIXAR TRANSCRIPTOMAS - FASTA
    # ======================================================
    transcriptoma_arabidopsis_fasta = (f"{OUTPUT_DIR}/arabidopsis_transcriptoma.fasta")
    baixar_transcriptoma( transcriptoma_arabidopsis_fasta, ARABIDOPSIS_QUERY)
    transcriptoma_tuberosum_fasta = (f"{OUTPUT_DIR}/tuberosum_transcriptoma.fasta")
    baixar_transcriptoma(transcriptoma_tuberosum_fasta, TUBEROSUM_QUERY)
    transcriptoma_physalis_fasta = (f"{OUTPUT_DIR}/physalis_transcriptoma.fasta")
    baixar_transcriptoma(transcriptoma_physalis_fasta, PHYSALIS_QUERY)
    
    predizer_transcritos(transcriptoma_arabidopsis_fasta, xgb_model,'XGBoost')
    predizer_transcritos(transcriptoma_tuberosum_fasta, xgb_model,'XGBoost')
    predizer_transcritos(transcriptoma_physalis_fasta, xgb_model,'XGBoost')


    predizer_transcritos(transcriptoma_arabidopsis_fasta, rf_model,"RF")
    predizer_transcritos(transcriptoma_tuberosum_fasta, rf_model,"RF")
    predizer_transcritos(transcriptoma_physalis_fasta, rf_model,"RF")

    predizer_transcritos(transcriptoma_arabidopsis_fasta, svm_model,"SVM")
    predizer_transcritos(transcriptoma_tuberosum_fasta, svm_model,"SVM")
    predizer_transcritos(transcriptoma_physalis_fasta, svm_model,"SVM")    
    
    print("\nArquivo salvo:")
    print(f"{OUTPUT_DIR}/predicoes.csv")



DOWNLOAD NCBI
Proteínas encontradas: 2260
Arquivo salvo: pipelineI/positive.fasta

DOWNLOAD NCBI
Proteínas encontradas: 5000
Arquivo salvo: pipelineI/negative.fasta

Executando CD-HIT...
Program: CD-HIT, V4.8.1 (+OpenMP), Aug 20 2021, 08:39:56
Command: cd-hit -i pipelineI/positive.fasta -o
         pipelineI/positive_nr.fasta -c 0.7 -n 5 -M 16000 -T 4

Started: Wed May 27 15:15:20 2026
                            Output                              
----------------------------------------------------------------
total seq: 2260
longest and shortest : 2595 and 51
Total letters: 1790660
Sequences have been sorted

Approximated minimal memory consumption:
Sequence        : 2M
Buffer          : 4 X 11M = 44M
Table           : 2 X 65M = 130M
Miscellaneous   : 0M
Total           : 177M

Table limit with the given memory limit:
Max number of representatives: 4000000
Max number of word counting entries: 1977856658

# comparing sequences from          0  to        376
---------- new table wit

<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>

<Figure size 500x500 with 0 Axes>